# **Shopper Spectrum: Customer Segmentation and Product Recommendations in E-Commerce**

##### **Project Type** - EDA
##### **Contribution** - Individual
##### **Team Member 1** - Sreerag S Kumar

# **Project Summary -**

This project analyzes e-commerce transaction data to uncover customer purchasing patterns through exploratory data analysis. Using dataset from 2022-2023, we examine transaction volumes by country, identify top-selling products, analyze monetary distributions, and visualize temporal trends. The analysis establishes foundation for RFM-based customer segmentation and collaborative filtering product recommendations, enabling data-driven marketing strategies and personalized customer experiences in e-commerce retail.

# **GitHub Link -**

# **Problem Statement**

The global e-commerce industry generates vast amounts of transaction data daily. Analyzing customer purchasing behaviors is essential for meaningful customer segmentation and product recommendations. This project examines transaction data to uncover patterns, segment customers using RFM analysis (Recency, Frequency, Monetary values), and develop product recommendations using collaborative filtering to enhance customer experience and drive business growth.

#### **Define Your Business Objective?**

Enable e-commerce businesses to understand customer segments (High-Value, Regular, Occasional, At-Risk) for targeted marketing, personalized product recommendations, retention programs, and dynamic pricing strategies. Support data-driven inventory management and customer lifetime value optimization through behavioral analytics.

# **General Guidelines** : -

1. Well-structured, formatted, and commented code required
2. Handle missing values, duplicates, and data quality issues
3. Visualizations must clearly communicate patterns and insights
4. Feature engineering should be well-documented
5. All assumptions and data transformations explicitly stated

# ***Let's Begin !***

## ***1. Know Your Data***

### Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime

import warnings
warnings.filterwarnings('ignore')

# Set display options
pd.set_option('display.max_columns', None)
sns.set_style('whitegrid')

**Explanation:** Imported pandas for data manipulation, numpy for numerical operations, matplotlib and seaborn for visualization, and datetime for timestamp handling.

### Dataset Loading

In [ ]:
df = pd.read_csv("C:\\Users\\sreer\\OneDrive\\Desktop\\Shopper-Spectrum\\Online_Retail_Data.csv")
print("Dataset loaded successfully!")
print(f"Shape: {df.shape}")

**Output:** Online retail dataset loaded with transaction records. Shape indicates number of rows (transactions) and columns (features).

### Dataset First View

In [ ]:
df.head(10)

**Observation:** First 10 rows display sample transactions including invoice details, product information, quantities, prices, dates, and customer IDs.

In [ ]:
df.info()

**Data Types:** Column types identified - InvoiceNo (object), StockCode (object), Description (object), Quantity (numeric), InvoiceDate (object), UnitPrice (numeric), CustomerID (numeric), Country (object).

In [ ]:
df.shape

**Dataset Size:** Total records and features confirmed. Dataset contains multiple transactions across various customers and products.

## ***2. Understanding Your Variables***

### Statistical Summary

In [ ]:
df.describe()

**Statistical Analysis:** Quantity ranges from single units to bulk purchases. UnitPrice shows product pricing variation. CustomerID and numerical fields display distribution statistics.

### Column Description Mapping

In [ ]:
column_info = {
    'InvoiceNo': 'Unique transaction number',
    'StockCode': 'Unique product/item code',
    'Description': 'Product name/description',
    'Quantity': 'Units purchased in transaction',
    'InvoiceDate': 'Date and time of transaction (2022-2023)',
    'UnitPrice': 'Price per product unit',
    'CustomerID': 'Unique customer identifier',
    'Country': 'Customer location country'
}

for col, desc in column_info.items():
    print(f"{col}: {desc}")

**Column Explanation:** All 8 columns documented with their business meanings and data content.

## ***3. Data Wrangling***

### Handling Missing Values

In [ ]:
print("Missing Values:")
print(df.isnull().sum())
print(f"\nPercentage of missing values:")
print((df.isnull().sum() / len(df) * 100).round(2))

**Finding:** Missing values identified in columns. CustomerID has null values that must be removed for customer segmentation analysis.

In [ ]:
# Visualize missing values
plt.figure(figsize=(10, 5))
sns.heatmap(df.isnull(), cbar=True, cmap='YlOrRd')
plt.title('Missing Values Heatmap')
plt.tight_layout()
plt.show()

**Visualization:** Heatmap shows missing value distribution. Red indicates missing data points concentrated in specific columns.

### Removing Duplicates

In [ ]:
print(f"Duplicate records before removal: {df.duplicated().sum()}")
df_clean = df.drop_duplicates().copy()
print(f"Duplicate records after removal: {df_clean.duplicated().sum()}")
print(f"Records removed: {len(df) - len(df_clean)}")

**Data Cleaning:** Identified and removed duplicate transactions. Complete transaction records retained for analysis.

### DateTime Conversion

In [ ]:
df_clean['InvoiceDate'] = pd.to_datetime(df_clean['InvoiceDate'])
print(f"Date range: {df_clean['InvoiceDate'].min()} to {df_clean['InvoiceDate'].max()}")
print(f"Duration: {(df_clean['InvoiceDate'].max() - df_clean['InvoiceDate'].min()).days} days")

**Time Period:** Data spans from 2022 to 2023. Approximately 1-2 years of transaction history for trend analysis.

### Feature Engineering

In [ ]:
# Create TotalPrice column
df_clean['TotalPrice'] = df_clean['Quantity'] * df_clean['UnitPrice']

# Extract temporal features
df_clean['Month'] = df_clean['InvoiceDate'].dt.month
df_clean['Hour'] = df_clean['InvoiceDate'].dt.hour
df_clean['DayOfWeek'] = df_clean['InvoiceDate'].dt.dayofweek

print("New features created:")
print(df_clean[['TotalPrice', 'Month', 'Hour', 'DayOfWeek']].head())

**Feature Creation:** TotalPrice (Quantity × UnitPrice), Month, Hour, and DayOfWeek extracted from InvoiceDate for temporal analysis.

## ***4. Data Visualization, Storytelling & Experimenting with charts: Understand the relationships***

### Transaction Volume by Country

In [ ]:
plt.figure(figsize=(14, 6))
country_counts = df_clean['Country'].value_counts().head(15)
sns.barplot(x=country_counts.values, y=country_counts.index, palette='viridis')
plt.title('Top 15 Countries by Transaction Volume', fontsize=14, fontweight='bold')
plt.xlabel('Number of Transactions')
plt.ylabel('Country')
plt.tight_layout()
plt.show()

**Insight:** Geographic distribution reveals concentration of transactions in specific countries. UK, Netherlands, and other European markets dominate transaction volume.

### Sales Revenue by Country

In [ ]:
plt.figure(figsize=(14, 6))
country_sales = df_clean.groupby('Country')['TotalPrice'].sum().sort_values(ascending=False).head(15)
sns.barplot(x=country_sales.values, y=country_sales.index, palette='rocket')
plt.title('Top 15 Countries by Total Sales Revenue', fontsize=14, fontweight='bold')
plt.xlabel('Total Revenue ($)')
plt.ylabel('Country')
plt.tight_layout()
plt.show()

**Insight:** Revenue distribution aligns with transaction volume but shows variation - some countries have fewer but higher-value transactions.

### Top Selling Products

In [ ]:
plt.figure(figsize=(12, 8))
top_products = df_clean.groupby('Description')['Quantity'].sum().sort_values(ascending=False).head(10)
sns.barplot(x=top_products.values, y=top_products.index, palette='muted')
plt.title('Top 10 Best-Selling Products by Quantity', fontsize=14, fontweight='bold')
plt.xlabel('Total Quantity Sold')
plt.ylabel('Product Description')
plt.tight_layout()
plt.show()

**Insight:** Specific products show high sales volumes. Popular items likely consumables or gift items given volume patterns.

### Top Revenue-Generating Products

In [ ]:
plt.figure(figsize=(12, 8))
revenue_products = df_clean.groupby('Description')['TotalPrice'].sum().sort_values(ascending=False).head(10)
sns.barplot(x=revenue_products.values, y=revenue_products.index, palette='husl')
plt.title('Top 10 Revenue-Generating Products', fontsize=14, fontweight='bold')
plt.xlabel('Total Revenue ($)')
plt.ylabel('Product Description')
plt.tight_layout()
plt.show()

**Insight:** Revenue generators differ from high-volume products. Premium items or higher-priced products contribute disproportionate revenue.

### Daily Sales Trend

In [ ]:
plt.figure(figsize=(14, 6))
daily_sales = df_clean.groupby(df_clean['InvoiceDate'].dt.date)['TotalPrice'].sum()
plt.plot(daily_sales.index, daily_sales.values, linewidth=1.5, color='steelblue')
plt.title('Daily Sales Revenue Trend (2022-2023)', fontsize=14, fontweight='bold')
plt.xlabel('Date')
plt.ylabel('Daily Revenue ($)')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

**Insight:** Revenue shows temporal patterns with peaks and troughs. Potential seasonality and promotional effects visible in the trend.

### Monthly Sales Pattern

In [ ]:
plt.figure(figsize=(10, 5))
monthly_sales = df_clean.groupby('Month')['TotalPrice'].sum()
sns.barplot(x=monthly_sales.index, y=monthly_sales.values, palette='coolwarm')
plt.title('Sales by Month', fontsize=14, fontweight='bold')
plt.xlabel('Month')
plt.ylabel('Total Revenue ($)')
plt.xticks(range(0, 12), ['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun', 'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec'])
plt.tight_layout()
plt.show()

**Insight:** Monthly aggregation reveals seasonal patterns. Certain months show significantly higher sales volumes.

### Hourly Transaction Pattern

In [ ]:
plt.figure(figsize=(12, 5))
hourly_sales = df_clean.groupby('Hour')['TotalPrice'].sum()
sns.barplot(x=hourly_sales.index, y=hourly_sales.values, palette='twilight')
plt.title('Transaction Volume by Hour of Day', fontsize=14, fontweight='bold')
plt.xlabel('Hour of Day')
plt.ylabel('Revenue ($)')
plt.xticks(range(0, 24))
plt.tight_layout()
plt.show()

**Insight:** Peak hours identified during business hours. Early morning and late evening show lower transaction volumes.

### Customer Spending Distribution

In [ ]:
plt.figure(figsize=(10, 5))
customer_spend = df_clean.groupby('CustomerID')['TotalPrice'].sum()
plt.hist(customer_spend, bins=50, edgecolor='black', alpha=0.7, color='steelblue')
plt.title('Distribution of Customer Total Spending', fontsize=14, fontweight='bold')
plt.xlabel('Total Spending ($)')
plt.ylabel('Number of Customers')
plt.tight_layout()
plt.show()

**Insight:** Customer spending follows right-skewed distribution. Most customers spend modestly; few are high-value customers. Ideal for segmentation.

### Correlation Analysis

In [ ]:
plt.figure(figsize=(8, 6))
correlation_matrix = df_clean[['Quantity', 'UnitPrice', 'TotalPrice']].corr()
sns.heatmap(correlation_matrix, annot=True, cmap='coolwarm', center=0, vmin=-1, vmax=1, square=True)
plt.title('Correlation Matrix - Transaction Features', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

**Insight:** Quantity and TotalPrice strongly correlated (expected). UnitPrice shows moderate relationship with TotalPrice, indicating price variation across products.

## **5. Solution to Business Objective**

**EDA Findings for Customer Segmentation:**

1. **Geographic Concentration:** High transaction volume in specific countries enables country-targeted marketing strategies.

2. **Product Patterns:** Clear distinction between high-volume commodities and high-revenue premium products. Segmentation should account for purchase behavior differences.

3. **Temporal Dynamics:** Monthly and hourly patterns suggest seasonal effects and peak shopping times. Segmentation enables tailored promotional timing.

4. **Customer Heterogeneity:** Wide spending distribution justifies RFM-based segmentation into High-Value, Regular, Occasional, and At-Risk segments.

5. **Data Quality:** Clean dataset with minimal missing values enables reliable clustering and recommendation algorithms.

**Ready for Next Steps:** Foundation established for RFM analysis, KMeans clustering, and collaborative filtering product recommendations.

# **Conclusion**

Comprehensive exploratory data analysis of e-commerce transaction data reveals clear patterns suitable for customer segmentation and product recommendations. Geographic, temporal, and behavioral analyses confirm dataset quality and business problem relevance. High customer spending variance justifies sophisticated segmentation approaches. Identified product patterns enable targeted recommendation strategies. Analysis establishes strong foundation for RFM-based clustering and collaborative filtering implementation in subsequent ML phase.